In [4]:
import pandas as pd
import pyarrow.parquet as pq
data_path = r"C:\projects_data\ml_pipeline\data\cmapss_production_ready_2d.parquet"
data  = pd.read_parquet(data_path)
data.info()
data.shape
data.empty
df = pq.ParquetFile(data_path)
print(df.metadata)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   engine_id     20631 non-null  int64  
 1   time_cycle    20631 non-null  int64  
 2   op_setting_1  20631 non-null  float64
 3   op_setting_2  20631 non-null  float64
 4   sensor_2      20631 non-null  float64
 5   sensor_3      20631 non-null  float64
 6   sensor_4      20631 non-null  float64
 7   sensor_6      20631 non-null  float64
 8   sensor_7      20631 non-null  float64
 9   sensor_8      20631 non-null  float64
 10  sensor_9      20631 non-null  float64
 11  sensor_11     20631 non-null  float64
 12  sensor_12     20631 non-null  float64
 13  sensor_13     20631 non-null  float64
 14  sensor_14     20631 non-null  float64
 15  sensor_15     20631 non-null  float64
 16  sensor_17     20631 non-null  int64  
 17  sensor_20     20631 non-null  float64
 18  sensor_21     20631 non-nu

In [5]:
import pyarrow.parquet as pq
import torch 
from torch.utils.data import DataLoader,Dataset
import pandas as pd
import numpy as np

class ParquetDataset(Dataset):
    def __init__(self,path,window_size=30):
        df = pd.read_parquet(path)
        self.window_size = window_size
        self.valid_indices = []
        feature_cols = [cols for cols in df.columns if cols not in ['engine_id','time_cycle','RUL']]
        self.features = df[feature_cols].astype(np.float32).values
        self.targets = df['RUL'].astype(np.float32).values
        #now need to fill the valid indices based on window_size
        #for this we need to find the each engine's max cycle value then only we could find the valid_indices for every engine 
        # need to do it with pandas and numpy #dont know how to write in pandas and numpy 
        #???
        offset = self.window_size-1
        valid_mask = df['engine_id']==df['engine_id'].shift(-offset)
        self.valid_indices = df.index[valid_mask].to_numpy()
        
    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self,idx):
        #this function will return the features and targets for idx specified
        start_row = self.valid_indices[idx]
        end_row = start_row+self.window_size
        features = self.features[start_row:end_row]
        targets = self.targets[end_row-1]
        tensored_features = torch.Tensor(features)
        tensored_targets = torch.tensor(targets)
        
        return tensored_features,tensored_targets
    
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np

# 1. The Core Architecture
class PredictiveMaintanceLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        # batch_first=True ensures input is [Batch, Sequence, Features]
        self.lstm = nn.LSTM(input_size=17, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        # Smashes the 64 hidden features down into 1 continuous RUL prediction
        self.linear = nn.Linear(64, 1)
        
    def forward(self, x):
        # out shape: [Batch, Sequence, Hidden] -> [64, 30, 64]
        out, (hn, cn) = self.lstm(x)
        
        # FATAL FLAW 1 FIX: Extract the final time-step
        # out[:, -1, :] means: All batches, the very last time step, all features
        # new shape: [64, 64]
        last_time_step = out[:, -1, :]
        
        # new shape: [64, 1]
        pred = self.linear(last_time_step)
        
        return pred

# (Assuming your CMAPSSDataset class is defined here as written previously)

if __name__ == "__main__":
    # --- Data Loading ---
    path = r"C:\projects_data\ml_pipeline\data\cmapss_production_ready_2d.parquet"
    dataset = ParquetDataset(path=path, window_size=30)
    print(f"Total valid 30-cycle windows available: {len(dataset)}")
    
    dataloader = DataLoader(
        dataset, 
        batch_size=64, 
        shuffle=True,       
        num_workers=0,      
        drop_last=True      
    )

    # --- Initialization ---
    lstm = PredictiveMaintanceLSTM()
    
    # AdamW is generally far superior to SGD for training LSTMs
    optim = torch.optim.AdamW(lstm.parameters(), lr=1e-3)
    epochs = 5
    
    # FATAL FLAW 2 FIX: Use Mean Squared Error for continuous Regression targets
    criterion = nn.MSELoss()
    
    # --- The 5-Step Training Loop ---
    for epoch in range(epochs):
        lstm.train() # Ensure layers like dropout are active
        
        for i, (inputs, targets) in enumerate(dataloader):
            # 1. Zero Gradients
            optim.zero_grad()
            
            # 2. Forward Pass
            # predictions shape: [64, 1]
            predictions = lstm(inputs)
            
            # FATAL FLAW 3 FIX: Squeeze predictions to match target shape [64]
            predictions = predictions.squeeze()
            
            # 3. Calculate Loss
            loss = criterion(predictions, targets)
            
            # 4. Backward Pass (Compute gradients & destroy graph)
            loss.backward()
            
            # 5. Optimizer Step (Adjust weights)
            optim.step()
            
            if i % 50 == 0:
                # We use loss.item() to safely extract the raw float without leaking memory
                print(f"Epoch {epoch} | Batch {i} | MSE Loss: {loss.item():.4f}")
        

Total valid 30-cycle windows available: 17731
Epoch 0 | Batch 0 | MSE Loss: 8318.6934
Epoch 0 | Batch 50 | MSE Loss: 7265.1802
Epoch 0 | Batch 100 | MSE Loss: 7493.7876
Epoch 0 | Batch 150 | MSE Loss: 7021.2593
Epoch 0 | Batch 200 | MSE Loss: 6696.9438
Epoch 0 | Batch 250 | MSE Loss: 5083.2168
Epoch 1 | Batch 0 | MSE Loss: 4850.6631
Epoch 1 | Batch 50 | MSE Loss: 4628.4131
Epoch 1 | Batch 100 | MSE Loss: 5231.0049
Epoch 1 | Batch 150 | MSE Loss: 4576.6240
Epoch 1 | Batch 200 | MSE Loss: 4103.6426
Epoch 1 | Batch 250 | MSE Loss: 3674.1514
Epoch 2 | Batch 0 | MSE Loss: 4036.5151
Epoch 2 | Batch 50 | MSE Loss: 3450.8875
Epoch 2 | Batch 100 | MSE Loss: 2420.3511
Epoch 2 | Batch 150 | MSE Loss: 3563.5913
Epoch 2 | Batch 200 | MSE Loss: 2867.8269
Epoch 2 | Batch 250 | MSE Loss: 3001.4944
Epoch 3 | Batch 0 | MSE Loss: 2813.1694
Epoch 3 | Batch 50 | MSE Loss: 2376.7056
Epoch 3 | Batch 100 | MSE Loss: 2161.9048
Epoch 3 | Batch 150 | MSE Loss: 2820.9236
Epoch 3 | Batch 200 | MSE Loss: 2549.9763


In [6]:
path =  r"C:\projects_data\ml_pipeline\data\cmapss_production_ready_2d.parquet"
dataset = ParquetDataset(path = path,window_size=30)
print(f"Total valid 30-cycle windows available: {len(dataset)}")
dataloader = DataLoader(
    dataset, 
    batch_size=64, 
    shuffle=True,       
    num_workers=0,      
    drop_last=True      
)


batch_features, batch_targets = next(iter(dataloader))

print("\n--- Batch Verification ---")
print(f"Features Shape: {batch_features.shape}") 
print(f"Targets Shape: {batch_targets.shape}")



Total valid 30-cycle windows available: 17731

--- Batch Verification ---
Features Shape: torch.Size([64, 30, 17])
Targets Shape: torch.Size([64])


In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np

# 1. The Core Architecture
class PredictiveMaintanceLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        # batch_first=True ensures input is [Batch, Sequence, Features]
        self.lstm = nn.LSTM(input_size=17, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        # Smashes the 64 hidden features down into 1 continuous RUL prediction
        self.linear = nn.Linear(64, 1)
        
    def forward(self, x):
        # out shape: [Batch, Sequence, Hidden] -> [64, 30, 64]
        out, (hn, cn) = self.lstm(x)
        
        # FATAL FLAW 1 FIX: Extract the final time-step
        # out[:, -1, :] means: All batches, the very last time step, all features
        # new shape: [64, 64]
        last_time_step = out[:, -1, :]
        
        # new shape: [64, 1]
        pred = self.linear(last_time_step)
        
        return pred

# (Assuming your CMAPSSDataset class is defined here as written previously)

if __name__ == "__main__":
    # --- Data Loading ---
    path = r"C:\projects_data\ml_pipeline\data\cmapss_production_ready_2d.parquet"
    dataset = CMAPSSDataset(path=path, window_size=30)
    print(f"Total valid 30-cycle windows available: {len(dataset)}")
    
    dataloader = DataLoader(
        dataset, 
        batch_size=64, 
        shuffle=True,       
        num_workers=0,      
        drop_last=True      
    )

    # --- Initialization ---
    lstm = PredictiveMaintanceLSTM()
    
    # AdamW is generally far superior to SGD for training LSTMs
    optim = torch.optim.AdamW(lstm.parameters(), lr=1e-3)
    epochs = 5
    
    # FATAL FLAW 2 FIX: Use Mean Squared Error for continuous Regression targets
    criterion = nn.MSELoss()
    
    # --- The 5-Step Training Loop ---
    for epoch in range(epochs):
        lstm.train() # Ensure layers like dropout are active
        
        for i, (inputs, targets) in enumerate(dataloader):
            # 1. Zero Gradients
            optim.zero_grad()
            
            # 2. Forward Pass
            # predictions shape: [64, 1]
            predictions = lstm(inputs)
            
            # FATAL FLAW 3 FIX: Squeeze predictions to match target shape [64]
            predictions = predictions.squeeze()
            
            # 3. Calculate Loss
            loss = criterion(predictions, targets)
            
            # 4. Backward Pass (Compute gradients & destroy graph)
            loss.backward()
            
            # 5. Optimizer Step (Adjust weights)
            optim.step()
            
            if i % 50 == 0:
                # We use loss.item() to safely extract the raw float without leaking memory
                print(f"Epoch {epoch} | Batch {i} | MSE Loss: {loss.item():.4f}")

NameError: name 'CMAPSSDataset' is not defined

In [ ]:
import pyarrow.parquet as pq
import torch 
from torch.utils.data import DataLoader,Dataset
import pandas as pd
import numpy as np

class ParquetDataset(Dataset):
    def __init__(self,path,window_size=30):
        df = pd.read_parquet(path)
        self.window_size = window_size
        self.valid_indices = []
        feature_cols = [cols for cols in df.columns if cols not in ['engine_id','time_cycle','RUL']]
        self.features = df[feature_cols].astype(np.float32).values
        self.targets = df['RUL'].astype(np.float32).values
        #now need to fill the valid indices based on window_size
        #for this we need to find the each engine's max cycle value then only we could find the valid_indices for every engine 
        # need to do it with pandas and numpy #dont know how to write in pandas and numpy 
        #???
        offset = self.window_size-1
        valid_mask = df['engine_id']==df['engine_id'].shift(-offset)
        self.valid_indices = df.index[valid_mask].to_numpy()
        
    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self,idx):
        #this function will return the features and targets for idx specified
        start_row = self.valid_indices[idx]
        end_row = start_row+self.window_size
        features = self.features[start_row:end_row]
        targets = self.targets[end_row-1]
        tensored_features = torch.Tensor(features)
        tensored_targets = torch.tensor(targets)
        
        return tensored_features,tensored_targets
    

path =  r"C:\projects_data\ml_pipeline\data\cmapss_production_ready_2d.parquet"
dataset = ParquetDataset(path = path,window_size=30)
print(f"Total valid 30-cycle windows available: {len(dataset)}")


train_size = int(0.8*len(dataset))
print(train_size)

train_dataset,validation_dataset = torch.utils.data.random_split(dataset,lengths=train_size)
train_loader = DataLoader(train_dataset) #since the split has been happended dont know exactly what to write the parameters
validation_loader = DataLoader(validation_dataset) #same like train_loader dont know what to write 





Total valid 30-cycle windows available: 17731
14184


In [ ]:
class ToTensor:
    def __call__(self,features,targets):
        features = torch.tensor(features,dtype = torch.float32)
        targets = torch.tensor(targets,dtype = torch.float32)
        
        return features,targets
    
class TargetNormalize:
    def __init__(self,max_rul):
        self.maxrul = max_rul
    def __call__(self,features,targets):
        targets = targets/self.maxrul
        return features,targets
    
class Compose:

    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, features, target):
        for transform in self.transforms:
            features, target = transform(features, target)
        return features, target
    
class ParquetDataset(Dataset):
    def __init__(self,path,window_size=30,transform = None):
        df = pd.read_parquet(path)
        self.window_size = window_size
        self.valid_indices = []
        self.transform = transform
        feature_cols = [cols for cols in df.columns if cols not in ['engine_id','time_cycle','RUL']]
        self.features = df[feature_cols].astype(np.float32).values
        self.targets = df['RUL'].astype(np.float32).values
        #now need to fill the valid indices based on window_size
        #for this we need to find the each engine's max cycle value then only we could find the valid_indices for every engine 
        # need to do it with pandas and numpy #dont know how to write in pandas and numpy 
        #???
        offset = self.window_size-1
        valid_mask = df['engine_id']==df['engine_id'].shift(-offset)
        self.valid_indices = df.index[valid_mask].to_numpy()
        
    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self,idx):
        #this function will return the features and targets for idx specified
        start_row = self.valid_indices[idx]
        end_row = start_row+self.window_size
        features = self.features[start_row:end_row]
        targets = self.targets[end_row-1]
        if self.transform is not None:
            features, targets = self.transform(features, targets)
        return features,targets
    

if __name__=="__main__":
    path =  r"C:\projects_data\ml_pipeline\data\cmapss_production_ready_2d.parquet"
    dataset = ParquetDataset(path = path,window_size=30)
    train_size = int(0.8*len(dataset))
    valid_size = len(dataset)-train_size
    train_dataset, validation_dataset = torch.utils.data.random_split(
        dataset,
        [train_size, valid_size]
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=32,
        shuffle=False
    )
    trainset_targets = dataset.targets[train_dataset.indices]
    max_rul = trainset_targets.max()
    transform = Compose([
        TargetNormalize(max_rul),
        ToTensor()
    ])
    dataset.transform = transform

    
    

    
    


In [ ]:
C:\projects_data\ml_pipeline\data\output_parquet.parquet
C:\projects_data\ml_pipeline\data\output_json.json